## Import and Clean Dataset

### 1. Import libraries

We import the libraries needed to request data from the API,
organize the data, randomly select positions, and add pauses between requests.

In [2]:
import requests
import pandas as pd
import random
import time

### 2. Set up the API and sampling plan

We use the Hugging Face API to collect 12,000 conversations.
We select random positions in the dataset and download up to 30 conversations
from each position. A random seed makes the selection reproducible.

In [3]:
# Hugging Face API address
url = "https://datasets-server.huggingface.co/rows"

# Sampling settings
total_rows = 837989
batch_size = 30
target_rows = 12000

# Set a seed so the same positions can be selected again
random.seed(42)

# Calculate how many batches are needed
number_of_batches = (target_rows + batch_size - 1) // batch_size

# Randomly select non-overlapping batches
batch_ids = random.sample(
    range(total_rows // batch_size),
    number_of_batches
)

print("Number of batches:", number_of_batches)

Number of batches: 400


### 3. Download conversations

We download conversations from the selected positions in the dataset.
We pause between requests to reduce the chance of sending too many requests.
If a request fails, we retry it and record any batches that still fail.

In [4]:
# Create empty lists to store conversations and failed batches
all_rows = []
failed_batches = []

# Go through each randomly selected batch
for i, batch_id in enumerate(batch_ids):

    # Calculate the starting position and number of rows
    offset = batch_id * batch_size
    length = min(batch_size, target_rows - i * batch_size)

    # Set the information needed for the API request
    params = {
        "dataset": "allenai/WildChat-1M",
        "config": "default",
        "split": "train",
        "offset": offset,
        "length": length
    }

    success = False

    # Try each request up to three times
    for attempt in range(3):
        try:
            # Request data from the API
            response = requests.get(url, params=params, timeout=30)

            # Wait longer if the server reports too many requests
            if response.status_code == 429:
                print(f"Batch {i + 1}: too many requests. Waiting 60 seconds...")
                time.sleep(60)
                continue

            # Check whether the request was successful
            response.raise_for_status()

            # Extract the rows from the API response
            rows = response.json()["rows"]

            # Add each conversation to all_rows
            for item in rows:
                all_rows.append(item["row"])

            success = True
            break

        except requests.exceptions.RequestException as e:
            # Show the error and wait before trying again
            print(f"Batch {i + 1}, attempt {attempt + 1} failed: {e}")
            time.sleep(5)

    # Save the starting position if all attempts failed
    if not success:
        failed_batches.append(offset)

    # Show progress every 20 batches
    if (i + 1) % 20 == 0 or i == number_of_batches - 1:
        print(f"Completed {i + 1}/{number_of_batches} batches")

    # Pause before sending the next request
    time.sleep(2)

Completed 20/400 batches
Batch 29: too many requests. Waiting 60 seconds...
Completed 40/400 batches
Batch 59: too many requests. Waiting 60 seconds...
Completed 60/400 batches
Batch 61, attempt 1 failed: 502 Server Error: Bad Gateway for url: https://datasets-server.huggingface.co/rows?dataset=allenai%2FWildChat-1M&config=default&split=train&offset=451620&length=30
Completed 80/400 batches
Batch 88: too many requests. Waiting 60 seconds...
Completed 100/400 batches
Batch 115: too many requests. Waiting 60 seconds...
Completed 120/400 batches
Completed 140/400 batches
Batch 146: too many requests. Waiting 60 seconds...
Completed 160/400 batches
Batch 180: too many requests. Waiting 60 seconds...
Completed 180/400 batches
Completed 200/400 batches
Batch 211: too many requests. Waiting 60 seconds...
Completed 220/400 batches
Batch 237: too many requests. Waiting 60 seconds...
Completed 240/400 batches
Completed 260/400 batches
Batch 265: too many requests. Waiting 60 seconds...
Completed

### 4. Create and check the DataFrame

We put the downloaded conversations into a pandas DataFrame.
Then, we check the number of conversations and any failed batches.

In [15]:
# Convert the downloaded conversations into a DataFrame
df = pd.DataFrame(all_rows)

# Check the results
print("Conversations downloaded:", len(df))
print("Failed batch offsets:", failed_batches)

# Display the first five rows
df.head()

Conversations downloaded: 12000
Failed batch offsets: []


,conversation_hash,model,timestamp,conversation,turn,language,openai_moderation,detoxify_moderation,toxic,redacted,state,country,hashed_ip,header
0,e0c5aec012f5acde923390707b58741f,gpt-4-1106-preview,2023-12-23T17:49:33Z,[{'content': 'Create a scam-baiting response t...,3,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.00013213619240559638, '...",False,True,Florida,United States,96468a552ddf93221173da70c4307bdd7039d76e2ec154...,"{'accept-language': None, 'user-agent': 'Mozil..."
1,44e3256dff0f1ed796130746df4dc155,gpt-3.5-turbo-0613,2023-12-23T17:50:21Z,"[{'content': 'Will this formula work, and what...",3,English,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0007662957068532705, 'i...",False,False,Oxfordshire,United Kingdom,e0f8f253fd649edfd6f2afc4e990d1d779321a306c3cb7...,"{'accept-language': 'en-US,en;q=0.9', 'user-ag..."
2,962abf7e81fcf713de545607b23ef599,gpt-3.5-turbo-0613,2023-12-23T17:51:21Z,"[{'content': '– Нет, Оксана Игоревна, – прогов...",1,Russian,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0012769330060109496, 'i...",False,False,Chelyabinsk Oblast,Russia,2eef7d0f0b5b3cdd22f40849eefe0f13b05730cffa5bfd...,"{'accept-language': 'ru-RU,ru;q=0.9,en-US;q=0...."
3,7998f2e098c55ab13a0cf3462a7fae98,gpt-3.5-turbo-0613,2023-12-23T17:52:12Z,"[{'content': '– Нет, Оксана Игоревна, – прогов...",1,Russian,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0012769330060109496, 'i...",False,False,Chelyabinsk Oblast,Russia,2eef7d0f0b5b3cdd22f40849eefe0f13b05730cffa5bfd...,"{'accept-language': 'ru-RU,ru;q=0.9,en-US;q=0...."
4,0258a1739483b7e196b8418c222af34e,gpt-3.5-turbo-0613,2023-12-23T17:52:41Z,"[{'content': '– Нет, Оксана Игоревна, – прогов...",1,Russian,"[{'categories': {'harassment': False, 'harassm...","[{'identity_attack': 0.0012769330060109496, 'i...",False,False,Chelyabinsk Oblast,Russia,2eef7d0f0b5b3cdd22f40849eefe0f13b05730cffa5bfd...,"{'accept-language': 'ru-RU,ru;q=0.9,en-US;q=0...."


### 5. Filter conversations

We keep English conversations with at least two non-empty user messages.
This allows us to compare the first and last user messages in each conversation.

In [16]:
# Count non-empty user messages in a conversation
def count_user_messages(conversation):
    count = 0

    for message in conversation:
        content = message.get("content")

        if message.get("role") == "user" and isinstance(content, str) and content.strip():
            count += 1

    return count


# Keep conversations labeled as English
english_df = df[df["language"] == "English"].copy()

# Count user messages in each English conversation
english_df["user_message_count"] = english_df["conversation"].apply(
    count_user_messages
)

# Keep conversations with at least two non-empty user messages
filtered_df = english_df[english_df["user_message_count"] >= 2].copy()

# Check how many conversations remain after each step
print("Original conversations:", len(df))
print("English conversations:", len(english_df))
print("English conversations with 2+ user messages:", len(filtered_df))

# Display the first five rows
filtered_df[["conversation_hash", "language", "user_message_count"]].head()

Original conversations: 12000
English conversations: 6694
English conversations with 2+ user messages: 2277


,conversation_hash,language,user_message_count
0,e0c5aec012f5acde923390707b58741f,English,3
1,44e3256dff0f1ed796130746df4dc155,English,3
5,0c340d1f331581cd317f8e61bf976bb5,English,14
9,f3398b15f0b83c944378c9bcd132e722,English,2
10,6aa2d2f2d2629969d36bff490dc81408,English,4


### 6. Check for duplicates

We check whether conversation IDs or complete conversation contents
are repeated before deciding how to handle duplicates.

In [17]:
import json

# Count repeated conversation IDs
duplicate_ids = filtered_df["conversation_hash"].duplicated().sum()

# Convert each conversation to a string so it can be compared
conversation_text = filtered_df["conversation"].apply(
    lambda conversation: json.dumps(conversation, sort_keys=True)
)

# Count exact duplicates across all message fields
duplicate_conversations = conversation_text.duplicated().sum()

print("Repeated conversation IDs:", duplicate_ids)
print("Exact duplicate conversations:", duplicate_conversations)

Repeated conversation IDs: 0
Exact duplicate conversations: 0


### 7. Check for repeated user messages

We compare the sequence of user messages in each conversation.
This helps us find conversations with the same user text,
even if other information is different.

In [18]:
# Extract the sequence of non-empty user messages
def get_user_messages(conversation):
    messages = []

    for message in conversation:
        content = message.get("content")

        if message.get("role") == "user" and isinstance(content, str) and content.strip():
            messages.append(content.strip())

    return tuple(messages)


# Create a column containing the user messages
filtered_df["user_messages"] = filtered_df["conversation"].apply(
    get_user_messages
)

# Count conversations with the same sequence of user messages
duplicate_user_text = filtered_df["user_messages"].duplicated().sum()

print("Conversations with repeated user text:", duplicate_user_text)

Conversations with repeated user text: 13


### 8. Remove repeated user messages

For the main analysis, we keep the first conversation from each set
with the same sequence of user messages. We also keep the original
filtered data so we can compare results later.

In [19]:
# Keep the first conversation with each sequence of user messages
analysis_df = filtered_df.drop_duplicates(
    subset="user_messages",
    keep="first"
).copy()

# Compare the number of conversations before and after removal
print("Before removing repeated user text:", len(filtered_df))
print("After removing repeated user text:", len(analysis_df))
print("Conversations removed:", len(filtered_df) - len(analysis_df))

Before removing repeated user text: 2277
After removing repeated user text: 2264
Conversations removed: 13


### 9. Extract the first and last user messages

We extract the first and last non-empty user messages from each conversation.
These messages will be used for the sentiment comparison.

In [20]:
# Extract the first user message
analysis_df["first_message"] = analysis_df["user_messages"].apply(
    lambda messages: messages[0]
)

# Extract the last user message
analysis_df["last_message"] = analysis_df["user_messages"].apply(
    lambda messages: messages[-1]
)

# Check the first five conversations
analysis_df[
    ["conversation_hash", "user_message_count", "first_message", "last_message"]
].head()

,conversation_hash,user_message_count,first_message,last_message
0,e0c5aec012f5acde923390707b58741f,3,Create a scam-baiting response to the most rec...,Create a scam-baiting response to Maria's most...
1,44e3256dff0f1ed796130746df4dc155,3,"Will this formula work, and what will it do: =...",I am getting a #SPILL! error
5,0c340d1f331581cd317f8e61bf976bb5,14,i want to install yersinia in my kali linux wi...,how to detele file in bin folder in kali linux
9,f3398b15f0b83c944378c9bcd132e722,2,how to givw permision for delete any file from...,chmod +t\nchmod: missing operand after ‘+t’\nT...
10,6aa2d2f2d2629969d36bff490dc81408,4,"glBindFramebuffer(GL_FRAMEBUFFER, fbo_id);\n\t...",#version 330 core\n\nin vec2 in_fUV;\n\nout ve...


## 10. Remove conversations containing possible API keys

Some conversations may contain text that looks like an OpenAI API key. We remove any conversation containing `sk-proj-` to avoid sharing possible keys in the dataset. We also record the number of conversations removed.

In [21]:
# Remove conversations that contain a possible OpenAI project API key
# Check each conversation for a possible OpenAI project API key
contains_key = analysis_df.apply(
    lambda row: "sk-proj-" in row.to_json(force_ascii=False),
    axis=1
)

# Count the conversations containing a possible key
print("Conversations removed:", contains_key.sum())

# Keep only conversations without this pattern
analysis_df = analysis_df.loc[~contains_key].copy()

# Show the number of conversations remaining
print("Conversations remaining:", len(analysis_df))

Conversations removed: 1
Conversations remaining: 2263


### 11. Save the cleaned data

We save the cleaned conversations as a JSON file so team members
can use the same data for further analysis.

In [22]:
# Save the cleaned data as a JSON file
analysis_df.to_json(
    "wildchat_cleaned.json",
    orient="records",
    force_ascii=False,
    indent=2
)

print("Cleaned data saved to wildchat_cleaned.json")
print("Number of conversations:", len(analysis_df))

Cleaned data saved to wildchat_cleaned.json
Number of conversations: 2263


### If wanna to use the cleaned data later u can use the code below:


In [13]:
import pandas as pd
analysis_df = pd.read_json("wildchat_cleaned.json")